# Patagonia: Attribution Models vs. Experiment

**Author:** Duy Nguyen  
**Date:** April 2026

---

This notebook analyzes Patagonia's digital marketing experiment to evaluate whether naive attribution models provide reliable guidance for marketing budget allocation, or whether experimentally-derived incremental lift is needed for sound decision-making.

Patagonia ran a 3-week holdout study across five channels - Email, YouTube, Instagram, Google Search, and Contextual Display - with 2 million randomly selected customers. Twenty percent of customers per channel were held out from receiving marketing, enabling causal estimation of each channel's true impact.

## Setup

In [2]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import ttest_ind

pd.set_option('display.float_format', '{:,.4f}'.format)

df = pd.read_csv('data/patagonia.csv')
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (2000000, 22)


,age,gender,days_since_last_visit,days_since_last_purchase,num_past_purchases,sales_past_12m,time_on_site_last_visit,time_on_site_12m,email_T,email_touchpoints,...,instagram_T,instagram_touchpoints,google_search_T,google_search_touchpoints,contextual_display_T,contextual_display_touchpoints,sales,touchpoint_sequence,first_touch,last_touch
0,29,Female,4.6900,7.9100,0,120.1300,4.3500,98.0800,1,1,...,0,0,0,0,1,1,0.0000,"email,contextual_display",email,contextual_display
1,41,Female,30.1000,40.7300,1,77.3000,13.2800,28.9000,1,0,...,1,0,1,1,1,1,0.0000,"contextual_display,google_search",contextual_display,google_search
2,41,Female,13.1700,43.5300,1,191.7000,1.2600,140.3700,1,0,...,1,2,0,0,1,0,0.0000,"instagram,instagram",instagram,instagram
3,46,Male,9.1300,26.2900,3,202.7300,0.0700,60.5100,0,0,...,1,1,1,0,1,1,0.0000,"contextual_display,instagram",contextual_display,instagram
4,50,Male,1.7000,9.1700,1,136.0600,15.4100,23.7600,1,1,...,0,0,0,0,1,0,0.0000,email,email,email


## Question 1: Attribution Without Incrementality

Before leveraging the experimental design, we apply three standard marketing attribution models to understand what they suggest about each channel's contribution to sales. These models allocate observed sales across the touchpoints in a customer's journey without accounting for what would have happened in the absence of marketing.

### 1a. Last-Touch Attribution

Last-touch attribution assigns 100% of the credit for a conversion to the final channel a customer interacted with before purchasing. This model is simple and widely used, but it systematically favors channels that appear late in the funnel - such as search - regardless of whether those channels were causally responsible for the purchase decision.

In [15]:
converters = df[df['sales'] > 0].copy()

last_touch = (
    converters.groupby('last_touch')['sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
last_touch.columns = ['Channel', 'Attributed Sales ($)']
last_touch['Share (%)'] = (last_touch['Attributed Sales ($)'] / last_touch['Attributed Sales ($)'].sum() * 100).round(2)
print('Last-Touch Attribution')
last_touch

Last-Touch Attribution


,Channel,Attributed Sales ($),Share (%)
0,contextual_display,"544,446.5600",36.8100
1,google_search,"369,961.5200",25.0200
2,instagram,"321,664.5500",21.7500
3,youtube,"139,696.0800",9.4500
4,email,"103,145.3500",6.9700


### 1b. First-Touch Attribution

First-touch attribution assigns 100% of the credit to the first channel a customer encountered. This model favors awareness-driving channels at the top of the funnel and is often used to justify prospecting spend. Like last-touch, it ignores the full conversion journey and attributes all value to a single touchpoint.

In [3]:
first_touch = (
    converters.groupby('first_touch')['sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
first_touch.columns = ['Channel', 'Attributed Sales ($)']
first_touch['Share (%)'] = (first_touch['Attributed Sales ($)'] / first_touch['Attributed Sales ($)'].sum() * 100).round(2)
print('First-Touch Attribution')
first_touch

First-Touch Attribution


,Channel,Attributed Sales ($),Share (%)
0,contextual_display,"665,701.1000",45.0100
1,email,"471,110.3800",31.8600
2,youtube,"166,966.6400",11.2900
3,instagram,"101,525.2000",6.8600
4,google_search,"73,610.7400",4.9800


### 1c. Linear Attribution

Linear attribution distributes credit proportionally across all channels a customer was exposed to, weighted by the number of touchpoints per channel. A customer who received 3 email touchpoints and 2 YouTube touchpoints would allocate 3/5 of their sales to Email and 2/5 to YouTube. This approach is more balanced than single-touch models but still does not distinguish between channels that drove the conversion and those that were merely present.

In [4]:
channels = ['email', 'youtube', 'instagram', 'google_search', 'contextual_display']
touchpoint_cols = [f'{ch}_touchpoints' for ch in channels]

total_touchpoints = converters[touchpoint_cols].sum(axis=1)
exposed = converters[total_touchpoints > 0].copy()
exposed_totals = total_touchpoints[total_touchpoints > 0]

linear_credits = {}
for ch in channels:
    col = f'{ch}_touchpoints'
    linear_credits[ch] = (exposed[col] / exposed_totals * exposed['sales']).sum()

linear_touch = (
    pd.DataFrame(list(linear_credits.items()), columns=['Channel', 'Attributed Sales ($)'])
    .sort_values('Attributed Sales ($)', ascending=False)
    .reset_index(drop=True)
)
linear_touch['Share (%)'] = (linear_touch['Attributed Sales ($)'] / linear_touch['Attributed Sales ($)'].sum() * 100).round(2)
print('Linear Attribution')
linear_touch

Linear Attribution


,Channel,Attributed Sales ($),Share (%)
0,contextual_display,"675,554.6806",45.6800
1,email,"247,008.4450",16.7000
2,instagram,"202,725.1228",13.7100
3,google_search,"184,190.4411",12.4500
4,youtube,"169,435.3705",11.4600


### 1d. Comparing the Three Models

The three models often produce meaningfully different rankings and credit allocations across channels. These differences reflect the models' structural assumptions rather than any underlying truth about marketing effectiveness.

The critical limitation of all three models is that they attribute 100% of observed sales to marketing - implicitly assuming that no purchases would have occurred organically. This leads to a fundamental overestimate of marketing-driven revenue and can produce misleading budget recommendations.

In [5]:
comparison = (
    last_touch[['Channel', 'Share (%)']].rename(columns={'Share (%)': 'Last-Touch (%)'})
    .merge(first_touch[['Channel', 'Share (%)']].rename(columns={'Share (%)': 'First-Touch (%)'}), on='Channel')
    .merge(linear_touch[['Channel', 'Share (%)']].rename(columns={'Share (%)': 'Linear (%)'}), on='Channel')
    .sort_values('Last-Touch (%)', ascending=False)
)
print('Attribution Model Comparison - Share of Sales (%)')
comparison

Attribution Model Comparison - Share of Sales (%)


,Channel,Last-Touch (%),First-Touch (%),Linear (%)
0,contextual_display,36.8100,45.0100,45.6800
1,google_search,25.0200,4.9800,12.4500
2,instagram,21.7500,6.8600,13.7100
3,youtube,9.4500,11.2900,11.4600
4,email,6.9700,31.8600,16.7000


## Question 2: Estimating Incremental Sales with Experimental Data

The holdout experiment allows us to move beyond attribution and measure the true causal effect of each channel. By randomly withholding marketing from 20% of customers per channel, Patagonia created a valid counterfactual: what would these customers have spent without the marketing exposure?

### 2a. Intent-to-Treat (ITT) Effects

The Intent-to-Treat effect measures the average sales difference between the treatment group (customers eligible to receive the channel) and the holdout group (customers withheld from the channel). We use two-sample t-tests to assess statistical significance.

In [6]:
channel_T = {
    'Email': 'email_T',
    'YouTube': 'youtube_T',
    'Instagram': 'instagram_T',
    'Google Search': 'google_search_T',
    'Contextual Display': 'contextual_display_T'
}

itt_results = []
for channel, t_col in channel_T.items():
    treatment = df[df[t_col] == 1]['sales']
    holdout   = df[df[t_col] == 0]['sales']
    itt = treatment.mean() - holdout.mean()
    t_stat, p_val = ttest_ind(treatment, holdout)
    itt_results.append({
        'Channel': channel,
        'Treatment Mean ($)': treatment.mean(),
        'Holdout Mean ($)': holdout.mean(),
        'ITT Effect ($)': itt,
        'p-value': p_val,
        'Significant': 'Yes' if p_val < 0.05 else 'No'
    })

itt_df = pd.DataFrame(itt_results)
print('Intent-to-Treat Effects by Channel')
itt_df

Intent-to-Treat Effects by Channel


,Channel,Treatment Mean ($),Holdout Mean ($),ITT Effect ($),p-value,Significant
0,Email,0.8843,0.8147,0.0696,0.0000,Yes
1,YouTube,0.8847,0.8130,0.0717,0.0000,Yes
2,Instagram,0.8795,0.8339,0.0456,0.0024,Yes
3,Google Search,0.8777,0.8411,0.0365,0.0149,Yes
4,Contextual Display,0.8739,0.8564,0.0175,0.2441,No


### 2b. Total Incremental Sales

To translate per-customer ITT effects into total incremental revenue, we multiply each channel's ITT by the size of its treatment group.

In [7]:
incremental_sales = []
for row in itt_results:
    t_col = channel_T[row['Channel']]
    n_treatment = (df[t_col] == 1).sum()
    total_incremental = row['ITT Effect ($)'] * n_treatment
    incremental_sales.append({
        'Channel': row['Channel'],
        'ITT Effect ($)': row['ITT Effect ($)'],
        'Treatment Group Size': n_treatment,
        'Total Incremental Sales ($)': total_incremental
    })

incremental_df = pd.DataFrame(incremental_sales)
print(f"Total Incremental Sales: ${incremental_df['Total Incremental Sales ($)'].sum():,.2f}")
incremental_df

Total Incremental Sales: $385,488.46


,Channel,ITT Effect ($),Treatment Group Size,Total Incremental Sales ($)
0,Email,0.0696,1599749,"111,348.0083"
1,YouTube,0.0717,1599148,"114,700.1412"
2,Instagram,0.0456,1601007,"73,037.1989"
3,Google Search,0.0365,1599734,"58,453.0359"
4,Contextual Display,0.0175,1599262,"27,950.0716"


### 2c. Comparing Naive Attribution to Incremental Sales

Naive attribution models attribute all observed sales among converting customers to the marketing channels in their journey. The gap between naive totals and incremental estimates represents organic sales - conversions that would have happened regardless of marketing.

In [8]:
naive_total = last_touch['Attributed Sales ($)'].sum()
incremental_total = incremental_df['Total Incremental Sales ($)'].sum()
organic_estimate = naive_total - incremental_total

print(f'Naive Attribution Total (Last-Touch): ${naive_total:,.2f}')
print(f'Incremental Sales Total:              ${incremental_total:,.2f}')
print(f'Estimated Organic Sales:              ${organic_estimate:,.2f}')
print(f'Naive Overstatement Multiple:         {naive_total / incremental_total:.1f}x')

Naive Attribution Total (Last-Touch): $1,478,914.06
Incremental Sales Total:              $385,488.46
Estimated Organic Sales:              $1,093,425.60
Naive Overstatement Multiple:         3.8x


### 2d. Organic Sales and Implications

The difference between naive attribution totals and experimentally measured incremental sales represents conversions that would have occurred in the absence of any of the five channels. Naive attribution models assign this organic revenue to marketing, creating the illusion of high effectiveness. This overstatement can lead companies to over-invest in channels that generate little to no incremental value.

## Question 3: Estimating Incremental Impact Using Regression

Regression provides a flexible framework for estimating the incremental impact of each channel while controlling for pre-existing customer differences.

### 3a. Simple OLS Regression (Treatment Indicators Only)

We begin with a baseline regression using only the five treatment indicators. The coefficients represent the average incremental sales per customer assigned to each channel - equivalent to the ITT estimates above.

In [13]:
model_simple = smf.ols(
    'sales ~ email_T + youtube_T + instagram_T + google_search_T + contextual_display_T',
    data=df
).fit()
print(model_simple.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     12.16
Date:                Tue, 07 Apr 2026   Prob (F-statistic):           8.24e-12
Time:                        09:44:30   Log-Likelihood:            -7.1166e+06
No. Observations:             2000000   AIC:                         1.423e+07
Df Residuals:                 1999994   BIC:                         1.423e+07
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                0.6778 

### 3b. Regression with Control Variables

Adding customer-level controls - demographics and prior engagement - can reduce residual variance and improve the precision of treatment effect estimates.

In [8]:
model_controls = smf.ols(
    'sales ~ email_T + youtube_T + instagram_T + google_search_T + contextual_display_T '
    '+ age + C(gender) + days_since_last_visit + days_since_last_purchase '
    '+ num_past_purchases + sales_past_12m + time_on_site_last_visit + time_on_site_12m',
    data=df
).fit()
print(model_controls.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     290.7
Date:                Tue, 07 Apr 2026   Prob (F-statistic):               0.00
Time:                        09:36:52   Log-Likelihood:            -7.1147e+06
No. Observations:             2000000   AIC:                         1.423e+07
Df Residuals:                 1999986   BIC:                         1.423e+07
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

### 3c. Regression with Interaction Term

We explore whether the effectiveness of Google Search varies with a customer's purchase history. Customers with more past purchases may be more likely to respond to search ads, given higher brand familiarity and purchase intent.

The results show that the interaction between Google Search and number of past purchases is not significant.

In [9]:
model_interaction = smf.ols(
    'sales ~ email_T + youtube_T + instagram_T + google_search_T + contextual_display_T '
    '+ age + C(gender) + days_since_last_visit + days_since_last_purchase '
    '+ num_past_purchases + sales_past_12m + time_on_site_last_visit + time_on_site_12m '
    '+ google_search_T:num_past_purchases',
    data=df
).fit()
print(model_interaction.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     269.9
Date:                Tue, 07 Apr 2026   Prob (F-statistic):               0.00
Time:                        09:37:16   Log-Likelihood:            -7.1147e+06
No. Observations:             2000000   AIC:                         1.423e+07
Df Residuals:                 1999985   BIC:                         1.423e+07
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

### 3d. Comparing Regression Results

We extract treatment coefficients across all three models to assess stability. Under well-executed random assignment, coefficients should remain consistent across specifications.

In [12]:
treatment_vars = ['email_T', 'youtube_T', 'instagram_T', 'google_search_T', 'contextual_display_T']

coef_comparison = pd.DataFrame({
    'Variable': treatment_vars,
    'Simple OLS': [model_simple.params.get(v, np.nan) for v in treatment_vars],
    'With Controls': [model_controls.params.get(v, np.nan) for v in treatment_vars],
    'With Interaction': [model_interaction.params.get(v, np.nan) for v in treatment_vars]
})
print('Treatment Coefficient Comparison Across Models')
coef_comparison

Treatment Coefficient Comparison Across Models


,Variable,Simple OLS,With Controls,With Interaction
0,email_T,0.0696,0.0709,0.0709
1,youtube_T,0.0717,0.0724,0.0724
2,instagram_T,0.0456,0.0455,0.0455
3,google_search_T,0.0365,0.0364,0.0317
4,contextual_display_T,0.0173,0.0170,0.0170


## Question 4: Calculating ROI of Marketing Channels

ROI requires pairing each channel's incremental revenue with its actual cost. We calculate total spend using provided cost-per-touchpoint rates.

### 4a. Channel Cost Calculations

Cost rates per channel:
- **Email:** $0.12 per email sent
- **YouTube:** $0.20 per completed 30-second view
- **Instagram:** $0.10 per in-stream view (3+ seconds)
- **Google Search:** $2.30 per click
- **Contextual Display:** $0.02 per viewable impression

In [11]:
cost_per_touchpoint = {
    'email': 0.12,
    'youtube': 0.20,
    'instagram': 0.10,
    'google_search': 2.30,
    'contextual_display': 0.02
}

cost_rows = []
for ch, rate in cost_per_touchpoint.items():
    total_touchpoints = df[f'{ch}_touchpoints'].sum()
    total_cost = total_touchpoints * rate
    cost_rows.append({
        'Channel': ch.replace('_', ' ').title(),
        'Cost per Touchpoint ($)': rate,
        'Total Touchpoints': total_touchpoints,
        'Total Cost ($)': total_cost
    })

cost_df = pd.DataFrame(cost_rows)
print(f"Total Marketing Spend: ${cost_df['Total Cost ($)'].sum():,.2f}")
cost_df

Total Marketing Spend: $1,164,122.12


,Channel,Cost per Touchpoint ($),Total Touchpoints,Total Cost ($)
0,Email,0.1200,568853,"68,262.3600"
1,Youtube,0.2000,388892,"77,778.4000"
2,Instagram,0.1000,501936,"50,193.6000"
3,Google Search,2.3000,402568,"925,906.4000"
4,Contextual Display,0.0200,2099068,"41,981.3600"


### 4b. ROI Based on Incremental Sales

ROI is calculated as: **(Incremental Sales - Cost) / Cost x 100%**. We use regression coefficients from the model with controls to estimate incremental revenue per channel.

In [14]:
channel_map = {
    'Email': 'email_T',
    'Youtube': 'youtube_T',
    'Instagram': 'instagram_T',
    'Google Search': 'google_search_T',
    'Contextual Display': 'contextual_display_T'
}

roi_rows = []
for cost_row in cost_rows:
    channel_name = cost_row['Channel']
    t_col = channel_map.get(channel_name)
    if t_col is None:
        continue
    n_treatment = (df[t_col] == 1).sum()
    coef = model_simple.params.get(t_col, 0)
    incremental_rev = coef * n_treatment
    cost = cost_row['Total Cost ($)']
    roi = (incremental_rev - cost) / cost * 100 if cost > 0 else np.nan
    roi_rows.append({
        'Channel': channel_name,
        'Incremental Sales ($)': incremental_rev,
        'Total Cost ($)': cost,
        'ROI (%)': roi
    })

roi_df = pd.DataFrame(roi_rows).sort_values('ROI (%)', ascending=False)
print('ROI Based on Incremental (Experimental) Attribution')
roi_df

ROI Based on Incremental (Experimental) Attribution


,Channel,Incremental Sales ($),Total Cost ($),ROI (%)
0,Email,"111,301.4919","68,262.3600",63.0496
1,Youtube,"114,668.8862","77,778.4000",47.4302
2,Instagram,"73,029.2563","50,193.6000",45.4952
4,Contextual Display,"27,737.0218","41,981.3600",-33.9301
3,Google Search,"58,422.3851","925,906.4000",-93.6902


### 4c. ROI Under Naive Attribution (Last-Touch)

For comparison, we compute ROI using last-touch attributed sales. This reflects what a company relying solely on attribution modeling would believe about each channel's return.

In [34]:
lt_dict = last_touch.set_index('Channel')['Attributed Sales ($)'].to_dict()
lt_dict['Contextual Display'] = lt_dict.pop('contextual_display')
lt_dict['Email'] = lt_dict.pop('email')
lt_dict['Google Search'] = lt_dict.pop('google_search')
lt_dict['Instagram'] = lt_dict.pop('instagram')
lt_dict['Youtube'] = lt_dict.pop('youtube')

naive_roi_rows = []
for cost_row in cost_rows:
    channel_name = cost_row['Channel']
    naive_sales = lt_dict.get(channel_name, 0)
    cost = cost_row['Total Cost ($)']
    roi = (naive_sales - cost) / cost * 100 if cost > 0 else np.nan
    naive_roi_rows.append({
        'Channel': channel_name,
        'Naive Attributed Sales ($)': naive_sales,
        'Total Cost ($)': cost,
        'Naive ROI (%)': roi
    })

naive_roi_df = pd.DataFrame(naive_roi_rows).sort_values('Naive ROI (%)', ascending=False)
print('ROI Based on Last-Touch Attribution')
naive_roi_df

ROI Based on Last-Touch Attribution


,Channel,Naive Attributed Sales ($),Total Cost ($),Naive ROI (%)
4,Contextual Display,"544,446.5600","41,981.3600","1,196.8769"
2,Instagram,"321,664.5500","50,193.6000",540.8477
1,Youtube,"139,696.0800","77,778.4000",79.6078
0,Email,"103,145.3500","68,262.3600",51.1014
3,Google Search,"369,961.5200","925,906.4000",-60.0433


### 4d. Incremental vs. Naive ROI Comparison

In [35]:
roi_compare = (
    roi_df[['Channel', 'ROI (%)']].rename(columns={'ROI (%)': 'Incremental ROI (%)'})
    .merge(naive_roi_df[['Channel', 'Naive ROI (%)']],on='Channel')
    .sort_values('Incremental ROI (%)', ascending=False)
)
print('ROI Comparison: Incremental vs. Last-Touch Attribution')
roi_compare

ROI Comparison: Incremental vs. Last-Touch Attribution


,Channel,Incremental ROI (%),Naive ROI (%)
0,Email,63.0496,51.1014
1,Youtube,47.4302,79.6078
2,Instagram,45.4952,540.8477
3,Contextual Display,-33.9301,"1,196.8769"
4,Google Search,-93.6902,-60.0433


## Question 5: Was Simple Good Enough?

### 5a. Channel Rankings: Incremental vs. Naive

In [36]:
incremental_rank = roi_df[['Channel', 'ROI (%)']].rename(columns={'ROI (%)': 'Incremental ROI (%)'}).reset_index(drop=True)
incremental_rank['Incremental Rank'] = incremental_rank['Incremental ROI (%)'].rank(ascending=False).astype(int)

naive_rank = naive_roi_df[['Channel', 'Naive ROI (%)']].reset_index(drop=True)
naive_rank['Naive Rank'] = naive_rank['Naive ROI (%)'].rank(ascending=False).astype(int)

synthesis = (
    incremental_rank.merge(naive_rank, on='Channel')
    .sort_values('Incremental Rank')
)
synthesis['Rank Change'] = synthesis['Naive Rank'] - synthesis['Incremental Rank']
print('Channel Ranking: Incremental vs. Naive Attribution')
synthesis

Channel Ranking: Incremental vs. Naive Attribution


,Channel,Incremental ROI (%),Incremental Rank,Naive ROI (%),Naive Rank,Rank Change
0,Email,63.0496,1,51.1014,4,3
1,Youtube,47.4302,2,79.6078,3,1
2,Instagram,45.4952,3,540.8477,2,-1
3,Contextual Display,-33.9301,4,"1,196.8769",1,-3
4,Google Search,-93.6902,5,-60.0433,5,0


### 5b. Discussion: Was Naive Good Enough?

The analysis reveals a substantial divergence between naive attribution and experimentally-derived incremental impact across all five channels.

**The scale of overstatement:** Last-touch attribution claims `$1,478,914` in marketing-driven sales. The experiment shows only `$385,488` was truly incremental — a **3.8x overstatement**. The remaining $1,093,426 represents organic sales that would have occurred without any of the five channels. All naive models — last-touch, first-touch, and linear — implicitly attribute this organic revenue to marketing, inflating perceived performance.

**The most misleading case — Contextual Display:** Under last-touch attribution, Contextual Display ranks #1 with an apparent ROI of 1,197%. Under incremental measurement, it ranks #4 with an ROI of **-34%**, and its ITT effect is not statistically significant (p = 0.244). This channel appears highly effective in naive models solely because it generates a large volume of impressions that happen to appear in converters' journeys — not because it causes purchases. A marketer relying on attribution would dramatically over-invest in this channel.

**Instagram is similarly overstated:** Instagram ranks #2 under last-touch (541% ROI) but drops to #3 under incremental measurement (45% ROI). While it does generate positive incremental returns, its naive ranking is inflated by its presence in many conversion paths.

**Email is the most understated channel:** Last-touch attribution ranks Email #4 with a 51% ROI. The experiment reveals it is actually the best-performing channel with a **63% incremental ROI** — the highest across all five. Attribution undervalues Email because it rarely appears as the last touchpoint before conversion, even when it initiates or sustains the purchase journey.

**YouTube is slightly overstated:** Last-touch attribution shows a 80% ROI for YouTube, while incremental measurement puts it at 47% — still a solid positive return and the #2 ranked channel, but attribution inflates it modestly.

**Google Search is negative under both approaches:** Google Search is the only channel where both naive and incremental methods agree on the direction — negative ROI. However, the magnitude differs substantially: -60% under last-touch versus **-93.7% under incremental measurement**. This is driven by its very high cost per click ($2.30), which far exceeds the incremental lift it generates. Attribution makes this channel appear less unprofitable than it actually is.

**Limitations of the incremental analysis:**
- The 20% holdout design provides reliable ITT estimates but not complier average treatment effects (CATE). Customers who were held out but would never have engaged with the channel are included in the holdout group, potentially diluting the estimated effect.
- Google Search used geo-targeting rather than customer-level randomization, introducing potential confounds from geographic variation in consumer behavior.
- The three-week observation window may not capture long-run brand effects from channels like YouTube, whose value may accrue over subsequent purchase cycles.
- Cross-channel spillovers are not modeled — a customer held out from Email may receive more attention from Instagram, making channel-level estimates partially confounded.

**Conclusion:** Naive attribution models are not good enough for marketing budget allocation at Patagonia. The rankings they produce are largely inverted relative to the true incremental picture — the channel that appears best (Contextual Display) actually destroys value, while the channel that appears worst (Email) generates the highest returns. Organizations that rely on attribution alone risk making systematically wrong investment decisions. Ongoing holdout experimentation is essential for measuring true incremental impact as channel dynamics evolve.